# comic_gpu -- Colab GPU worker for comic-translator

**No ngrok, no API key.** Your PC and this notebook share the Drive folder `My Drive/comic_worker/`.
The PC posts jobs there (`comic colab analyze ...`, the cleaning stage's LaMa jobs); the last
cell picks them up, runs them on the T4 and writes the results back.

1. *Runtime > Change runtime type > T4 GPU*.
2. *Runtime > Run all*. Allow Drive access when asked.
3. Leave the tab open while the PC works. The last cell keeps running -- that is the worker.

Before the first run, on the PC: `python scripts/comic_run.py colab push` (sends the code; Colab downloads the models itself).
After code changes on the PC: `colab push`, then Run all again.


In [ ]:
# 1. GPU + Drive, and wait until everything `comic colab push` sent has arrived
from google.colab import drive
import os, json, time
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/comic_worker'   # created by the PC's `comic colab push`, never here
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

def wait_for_push(minutes=30):
    # Never create ROOT here: a folder made on this side before the PC's upload
    # lands becomes a second folder of the same name. Drive for desktop uploads in its own order and this mount sees new folders
    # late; remount until every file in pushed.json is here at its full size.
    deadline = time.time() + minutes * 60
    while True:
        missing = None
        try:
            listed = json.load(open(ROOT + '/pushed.json'))['files']
            missing = [k for k, size in listed.items()
                       if not os.path.exists(f'{ROOT}/{k}') or os.path.getsize(f'{ROOT}/{k}') != size]
        except (OSError, ValueError, KeyError):
            pass
        if missing == []:
            print(f'all {len(listed)} pushed files are here')
            return
        if time.time() > deadline:
            raise RuntimeError('push never arrived -- run `comic colab push` on the PC, keep Google Drive open')
        print('waiting for Drive upload:', 'no pushed.json yet' if missing is None
              else f'{len(missing)} files left, e.g. {missing[0]}', flush=True)
        time.sleep(45)
        drive.flush_and_unmount()
        drive.mount('/content/drive', force_remount=True)

wait_for_push()


In [ ]:
# 2. Dependencies (about a minute), pinned to the PC's versions so OCR reads the same.
#    - RapidOCR 1.4.4 (the PC's) declares Python <3.13 but is pure Python; without it pip silently
#      picked an OLDER RapidOCR on Colab's 3.13 and English OCR came out worse (2026-11-08).
#      So it is installed with --ignore-requires-python and its dependencies are listed by hand
#      (onnxruntime-gpu stands in for the CPU onnxruntime it would pull).
#    - Colab ships OpenCV 4.x; every OpenCV build is removed first and exactly one version installed.
#    The check runs in a fresh process (and so does the worker, cell 5), so no session restart is needed.
!pip -q uninstall -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless onnxruntime onnxruntime-gpu rapidocr-onnxruntime
!pip -q install "opencv-python==5.0.0.93" "opencv-python-headless==5.0.0.93" "shapely==2.2.0" "manga-ocr==0.1.16" pyclipper six tqdm pillow pydantic typer rich pyyaml fonttools scipy huggingface_hub onnxruntime-gpu
!pip -q install --no-deps --ignore-requires-python "rapidocr-onnxruntime==1.4.4"
!pip -q uninstall -y onnxruntime   # in case anything pulled the CPU build back in
!python -c "import cv2, numpy, onnxruntime as ort, importlib.metadata as md; v=md.version('rapidocr-onnxruntime'); print('cv2', cv2.__version__, '| numpy', numpy.__version__, '| rapidocr', v, '|', ort.get_available_providers()); assert cv2.__version__.startswith('5.0') and v == '1.4.4', 'versions differ from the PC'"


In [ ]:
# 3. Code: the project's comic package, pushed by `comic colab push`.
import sys, shutil
CODE = '/content/comic_code'
shutil.rmtree(CODE, ignore_errors=True)
shutil.copytree(ROOT + '/code', CODE)
if CODE not in sys.path:
    sys.path.insert(0, CODE)
for name in [m for m in sys.modules if m == 'comic' or m.startswith('comic.')]:
    del sys.modules[name]
print('code ready')


In [ ]:
# 4. Models: downloaded once by Colab into My Drive/comic_worker/models (fast datacenter link),
#    then copied to local disk each session. Nothing is uploaded from the PC.
import os, shutil, subprocess
from pathlib import Path
from huggingface_hub import hf_hub_download

drive_models = Path(ROOT) / 'models'
drive_models.mkdir(exist_ok=True)
env = dict(os.environ, COMIC_HOME=ROOT, PYTHONPATH=CODE)   # fetch_models writes to $COMIC_HOME/models
for only in ('comictextdetector', 'korean'):                # detection + Korean OCR; skips files already there
    subprocess.run(['python', f'{CODE}/scripts/fetch_models.py', '--only', only], env=env, check=True)
if not (drive_models / 'anime_manga_lama.pt').exists():     # Anime/Manga Big-LaMa (TorchScript)
    hf_hub_download('TareHimself/AnimeMangaInpainting-torchscript', 'anime_manga_lama.pt',
                    local_dir=str(drive_models))
local_models = Path('/content/models')
local_models.mkdir(exist_ok=True)
for f in drive_models.iterdir():
    if f.is_file():
        t = local_models / f.name
        if not t.exists() or t.stat().st_size != f.stat().st_size:
            shutil.copyfile(f, t)
print(sorted(p.name for p in local_models.iterdir()))


In [ ]:
# 5. Worker -- keeps running; stop it with the stop button.
#    A separate process, so it always loads the libraries cell 2 just installed.
import subprocess, sys
proc = subprocess.Popen(
    [sys.executable, '-u', '-c',
     f'import sys; sys.path.insert(0, {CODE!r}); from comic.colab.worker import main; main({ROOT!r})'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
try:
    for line in proc.stdout:
        print(line, end='', flush=True)
except KeyboardInterrupt:
    pass
finally:
    proc.terminate()
    print('worker stopped')
